In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

data = np.load("../data/processed/prepared.npz")
X_test, y_test = data["X_test"], data["y_test"]

logreg = joblib.load("../models/baseline_logreg.joblib")
rf = joblib.load("../models/random_forest.joblib")

print("Модели загружены. Test:", X_test.shape)

In [1]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, name in [(axes[0], logreg, "LogReg"),
                        (axes[1], rf, "RandomForest")]:
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Normal", "Attack"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    fn = cm[1, 0]
    ax.set_title(f"{name}: пропущено атак (FN) = {fn}")

plt.tight_layout()
plt.savefig("../reports/06_confusion_matrices.png", dpi=150)
plt.show()

NameError: name 'plt' is not defined

In [ ]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

rows = []
for t in np.arange(0.05, 0.96, 0.05):
    y_pred_t = (y_proba_rf >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    rows.append({
        "порог": round(t, 2), "TP": tp, "FP": fp, "пропущено_атак": fn,
        "precision": round(tp/(tp+fp), 4),
        "recall": round(tp/(tp+fn), 4),
        "f1": round(2*tp/(2*tp+fp+fn), 4),
    })

thr_table = pd.DataFrame(rows)
print(thr_table.to_string(index=False))

thr_table.plot(x="порог", y=["precision", "recall", "f1"], figsize=(10, 5), grid=True)
plt.title("Зависимость метрик от порога (Random Forest)")
plt.ylabel("Метрика")
plt.savefig("../reports/07_threshold_curve.png", dpi=150)
plt.show()

In [ ]:
best = thr_table.sort_values("f1", ascending=False).iloc[0]
print("Лучший порог по F1:")
print(best)

t_best = float(best["порог"])
y_pred_best = (y_proba_rf >= t_best).astype(int)
cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

print(f"\nПорог = {t_best}: поймано атак {tp}, пропущено {fn}, ложных тревог {fp}")

import json
with open("../reports/final_threshold.json", "w", encoding="utf-8") as f:
    json.dump({"model": "RandomForest",
               "threshold": t_best,
               "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)},
              f, indent=2, ensure_ascii=False)
print("Сохранено: reports/final_threshold.json")